# 🚀 Universal AI Workspace for Google Colab
### Production-Grade • Shareable • Multi-Mode Storage • Multi-Backend Inference • Secure Cloudflare Tunneling

This notebook serves as the thin, lightweight controller for your Universal AI Workspace.
- **100% Shareable & GitHub-Ready**: Code is decoupled from personal Google Drive and can run in `local_ssd` mode with zero Drive dependencies.
- **Multi-Source Code Bootstrap**: Load source code from GitHub, Google Drive, direct URL, or local storage into fast local SSD (`/content/app`).
- **True Multi-Mode I/O**: Supports Hugging Face, CivitAI, Google Drive shares (`gshare://` via `gdown`), direct URLs, and shallow git repos.
- **Zero-Degradation CUDA Acceleration**: Fully offloaded GGUF inference on Tesla T4/L4 GPUs (>> 50 tok/s on 0.5B models).

In [ ]:
#@title 1. Multi-Source Code & Storage Bootstrap Form
CODE_SOURCE = "google_drive" #@param ["github", "google_drive", "gdrive_relative", "url_zip", "direct_local"]
CODE_LOCATION = "AI_Workspace" #@param {type:"string"}
WORKSPACE_MODE = "hybrid" #@param ["hybrid", "local_ssd", "google_drive", "gdrive_relative", "custom_path"]
WORKSPACE_PATH = "AI_Workspace" #@param {type:"string"}

import os
import shutil
import subprocess
import sys
from pathlib import Path

LOCAL_APP_DIR = Path("/content/app")
LOCAL_APP_DIR.mkdir(parents=True, exist_ok=True)

# 1. Mount Google Drive if required by code source or storage mode
needs_drive = (CODE_SOURCE in ["google_drive", "gdrive_relative"]) or (WORKSPACE_MODE in ["hybrid", "google_drive", "gdrive_relative"])
if needs_drive:
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").is_dir():
            print("Mounting Google Drive...")
            drive.mount("/content/drive")
        else:
            print("Google Drive already mounted.")
    except Exception as e:
        print(f"Note: Drive mount not available ({e}). Proceeding...")

# 2. Acquire workspace code into fast local SSD /content/app to avoid Google Drive FUSE latency
if CODE_SOURCE == "github":
    repo_url = CODE_LOCATION if (CODE_LOCATION.startswith("http") or CODE_LOCATION.startswith("git")) else f"https://github.com/{CODE_LOCATION}.git"
    print(f"Cloning repository {repo_url} into {LOCAL_APP_DIR}...")
    if not (LOCAL_APP_DIR / "workspace_core").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(LOCAL_APP_DIR)], check=True)
elif CODE_SOURCE in ["google_drive", "gdrive_relative"]:
    drive_src = Path("/content/drive/MyDrive") / CODE_LOCATION.strip().lstrip("/")
    if drive_src.is_dir() and (drive_src / "workspace_core").is_dir():
        print(f"Syncing code from Google Drive to fast local SSD {LOCAL_APP_DIR}...")
        for item in ["workspace_core", "configs", "pyproject.toml", "README.md"]:
            s = drive_src / item
            d = LOCAL_APP_DIR / item
            if s.is_dir():
                shutil.copytree(str(s), str(d), dirs_exist_ok=True)
            elif s.is_file():
                shutil.copy2(str(s), str(d))
elif CODE_SOURCE == "url_zip":
    print(f"Downloading and extracting source zip from {CODE_LOCATION}...")
    import urllib.request, zipfile
    zip_tmp = Path("/content/app_source.zip")
    urllib.request.urlretrieve(CODE_LOCATION, str(zip_tmp))
    with zipfile.ZipFile(str(zip_tmp), "r") as z:
        z.extractall(str(LOCAL_APP_DIR))
elif CODE_SOURCE == "direct_local":
    loc_src = Path(CODE_LOCATION).resolve()
    if loc_src != LOCAL_APP_DIR.resolve() and (loc_src / "workspace_core").is_dir():
        for item in ["workspace_core", "configs", "pyproject.toml"]:
            s = loc_src / item
            if s.is_dir():
                shutil.copytree(str(s), str(LOCAL_APP_DIR / item), dirs_exist_ok=True)

# 3. Add fast local app to sys.path
if str(LOCAL_APP_DIR) not in sys.path:
    sys.path.insert(0, str(LOCAL_APP_DIR))
if "/content/drive/MyDrive/AI_Workspace" not in sys.path:
    sys.path.append("/content/drive/MyDrive/AI_Workspace")

# 4. Bootstrap workspace environment and native binaries
from workspace_core import get_config, EnvironmentManager, get_hardware_profile

config = get_config(workspace_mode=WORKSPACE_MODE, workspace_path=WORKSPACE_PATH)
env_mgr = EnvironmentManager(config)

print("Installing / verifying system tools and native binaries...")
env_mgr.ensure_system_packages()
env_mgr.fetch_cloudflared()
env_mgr.fetch_llama_cpp()
env_mgr.install_profile("core")

print("\n🚀 Universal AI Workspace Bootstrap Complete! Ready to run.")

In [ ]:
#@title 2. Hardware Diagnostics & Dynamic Environment Profiler
from workspace_core import get_hardware_profile
from rich.console import Console
from rich.table import Table

hw = get_hardware_profile()
console = Console()

table = Table(title="🖥️ Hardware & Environment Telemetry")
table.add_column("Component", style="cyan", no_wrap=True)
table.add_column("Details", style="green")

for k, v in hw.summary().items():
    table.add_row(k, str(v))

console.print(table)

In [ ]:
#@title 3. Quick Configuration & Interactive Dashboard Launcher
#@markdown Choose whether to launch the full 6-tab interactive management dashboard:
LAUNCH_INTERACTIVE_DASHBOARD = True #@param {type:"boolean"}
DASHBOARD_PORT = 7860 #@param {type:"integer"}
SHARE_PUBLIC_DASHBOARD = False #@param {type:"boolean"}

if LAUNCH_INTERACTIVE_DASHBOARD:
    from workspace_core.dashboard import launch_dashboard
    print(f"Launching Universal AI Workspace Dashboard on port {DASHBOARD_PORT}...")
    launch_dashboard(port=DASHBOARD_PORT, share=SHARE_PUBLIC_DASHBOARD)

In [ ]:
#@title 4. Headless 1-Click Quick-Start (Optional Alternative to Dashboard)
#@markdown Automatically inspect, download/load matching quant, start inference server & tunnel:
MODEL_SOURCE = "Qwen/Qwen2.5-0.5B-Instruct-GGUF" #@param {type:"string"}
INPUT_MODE = "auto" #@param ["auto", "huggingface", "direct", "google_drive", "gdrive_relative", "gdrive_share", "url", "civitai"]
PRESET = "gguf_balanced" #@param ["gguf_balanced", "gguf_long_ctx", "vlm_multimodal", "transformers_4bit", "vllm_high_throughput"]
CONTEXT_SIZE = 4096 #@param {type:"integer"}
ENABLE_CLOUDFLARE_TUNNEL = True #@param {type:"boolean"}
ENABLE_API_KEY_AUTH = True #@param {type:"boolean"}
OUTPUT_MODE = "hybrid_drive" #@param ["hybrid_drive", "local_ssd", "gdrive_relative", "huggingface_upload", "browser_download"]

from workspace_core import (
    get_config,
    ModelInspector,
    AutoSelector,
    Downloader,
    ModelRegistry,
    ProcessManager,
)

cfg = get_config()
inspector = ModelInspector()
selector = AutoSelector(config=cfg)
downloader = Downloader(cfg)
registry = ModelRegistry(cfg)
pm = ProcessManager(cfg)

print(f"[1/4] Inspecting {MODEL_SOURCE} (mode: {INPUT_MODE})...")
insp = inspector.inspect(MODEL_SOURCE, input_mode=INPUT_MODE)
rec = selector.recommend(insp, requested_ctx=CONTEXT_SIZE)
print(f"      Recommended Quant: {rec.recommended_quant} ({rec.reasoning})")

sync_to_drive = (OUTPUT_MODE in ["hybrid_drive", "gdrive_relative"]) and (getattr(cfg, "mode", "") != "local_ssd")
print(f"[2/4] Resolving / Downloading model weights (sync_to_drive={sync_to_drive})...")
dl = downloader.resolve_or_download(insp, variant=rec.selected_variant, sync_to_drive=sync_to_drive)
final_path = dl.primary_file_path

if rec.needs_requant and rec.requant_target_quant:
    print(f"      Re-quantizing to {rec.requant_target_quant}...")
    final_path = AutoSelector.execute_requantize(final_path, rec.requant_target_quant, backup_to_drive=sync_to_drive)

cat_map = {
    "llm-gguf": cfg.models_gguf_dir,
    "vlm-gguf": cfg.models_gguf_dir,
    "embedding-gguf": cfg.models_gguf_dir,
    "llm-safetensors": cfg.models_safetensors_dir,
    "diffusion": cfg.models_diffusion_dir,
    "audio": cfg.models_audio_dir,
}
dest_dir = cat_map.get(insp.model_type, cfg.models_gguf_dir)
drive_path = str(dest_dir / final_path.name) if sync_to_drive else None
shard_paths = [str(p) for p in dl.all_downloaded_files if p.resolve() != final_path.resolve()]

entry = registry.register_model(
    model_id=dl.model_id,
    name=dl.model_id,
    type=insp.model_type,
    source_url=insp.raw_source,
    quantization=rec.recommended_quant,
    total_size_bytes=dl.total_size_bytes,
    local_path=str(final_path),
    drive_path=drive_path,
    is_synced_to_drive=sync_to_drive,
    mmproj_path=str(dl.mmproj_file_path) if dl.mmproj_file_path else None,
    shard_files=shard_paths,
    set_active=True
)

print(f"[3/4] Launching server with preset \x27{PRESET}\x27...")
pm.start_server(model_id=entry.model_id, preset_name=PRESET, port=8000, enable_auth=ENABLE_API_KEY_AUTH, recommendation=rec)

pub_url = None
if ENABLE_CLOUDFLARE_TUNNEL:
    print("[4/4] Launching Cloudflare Tunnel...")
    pub_url = pm.start_tunnel(port=8000)

state = pm.get_state()
print("\n=========================================")
print(f"🟢 Server Status: {\x27RUNNING\x27 if state.is_server_running else \x27FAILED\x27}")
print(f"🔑 API Key:       {state.api_key or \x27None (Open Access)\x27}")
print(f"🌐 Local API:     http://127.0.0.1:{state.port}/v1")
if state.public_url:
    print(f"🚇 Public Tunnel: {state.public_url}/v1")
print("=========================================")

In [ ]:
#@title 5. Live Server Verification & Ready-to-Copy Client Examples
from workspace_core import ProcessManager, get_config
import requests
import json

cfg = get_config()
pm = ProcessManager(cfg)
state = pm.get_state()

if not state.is_server_running:
    print("⚠️ Server is not currently running. Please run Cell 3 or Cell 4 first.")
else:
    base_url = f"http://127.0.0.1:{state.port}"
    headers = {}
    if state.api_key:
        headers["Authorization"] = f"Bearer {state.api_key}"
    
    # 1. Test /v1/models
    try:
        r_models = requests.get(f"{base_url}/v1/models", headers=headers, timeout=5)
        print("✅ /v1/models response:", r_models.json())
    except Exception as e:
        print("❌ /v1/models test failed:", e)
    
    # 2. Test /v1/chat/completions (streaming)
    print("\nTesting streaming chat completions:")
    payload = {
        "model": state.active_model or "default",
        "messages": [{"role": "user", "content": "Say \x27Universal AI Workspace is ready!\x27 in 5 words."}],
        "stream": True,
        "temperature": 0.2
    }
    try:
        r_chat = requests.post(f"{base_url}/v1/chat/completions", json=payload, headers=headers, stream=True, timeout=30)
        for line in r_chat.iter_lines():
            if line:
                dec = line.decode("utf-8")
                if dec.startswith("data: "):
                    d = dec[6:].strip()
                    if d == "[DONE]": break
                    chunk = json.loads(d)
                    content = chunk.get("choices", [{}])[0].get("delta", {}).get("content") or ""
                    print(content, end="", flush=True)
        print("\n\n✅ Chat completions test passed!")
    except Exception as e:
        print("❌ Chat test failed:", e)
    
    # 3. Print copyable code snippets
    from workspace_core.dashboard import WorkspaceDashboard
    dash = WorkspaceDashboard(cfg)
    py_code, curl_code = dash.generate_code_snippets(state)
    print("\n" + "="*40 + " PYTHON CLIENT SNIPPET " + "="*40)
    print(py_code)
    print("="*40 + " CURL SNIPPET " + "="*40)
    print(curl_code)